# Agent: People (Phase 1, no MQTT)

This notebook implements a minimal local simulation for 50 people around Brøndby Stadium.

Phase 1 scope:
- One agent only
- Basic movement and state transitions
- No MQTT publish/subscribe
- Configuration loaded with `simulated_city.config.load_config()`

In [1]:
# Cell purpose: import required modules and load project configuration
from __future__ import annotations

from dataclasses import dataclass
from datetime import datetime, timezone
import math
import random
import sys
import os
from pathlib import Path
from typing import Literal

for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    src_dir = candidate / "src"
    if (src_dir / "simulated_city").exists():
        if str(src_dir) not in sys.path:
            sys.path.insert(0, str(src_dir))
        break

os.environ.setdefault("SIMCITY_MQTT_PROFILES", "mqtthq")

from simulated_city.config import load_config
import simulated_city.mqtt as mqtt

cfg = load_config()
sim_cfg = cfg.simulation
mqtt_cfg = cfg.mqtt

print(f"Loaded config. MQTT base topic: {cfg.mqtt.base_topic}")
print(f"Simulation section present: {sim_cfg is not None}")

Loaded config. MQTT base topic: simulated-city/stadium
Simulation section present: True


In [2]:
# Cell purpose: define local data model for people state used in Phase 1
State = Literal["approaching_entry", "inside", "exiting", "exited"]

@dataclass
class PersonState:
    person_id: str
    name: str
    lat: float
    lon: float
    state: State
    color: Literal["white", "green", "red"]
    speed_mps: float
    target_entry_id: str
    target_exit_id: str

print("Defined PersonState dataclass for local simulation state.")

Defined PersonState dataclass for local simulation state.


In [3]:
# Cell purpose: set simulation constants and build entry/exit points from config with safe fallbacks
TOTAL_PEOPLE = sim_cfg.total_people if sim_cfg else 50
TICKS = 180
TICK_SECONDS = sim_cfg.tick_interval_s if sim_cfg else 1.0
ENTRY_PROXIMITY_M = sim_cfg.entry_proximity_m if sim_cfg else 8.0
EXIT_REACHED_M = sim_cfg.exit_reached_m if sim_cfg else 8.0
TRUE_ALLOW_PROBABILITY = sim_cfg.true_allow_probability if sim_cfg else 0.80
SEED = sim_cfg.seed if sim_cfg and sim_cfg.seed is not None else 42

# Fallback center near Brøndby Stadium if no simulation locations are configured
center_lat = sim_cfg.center_lat if sim_cfg else 55.6479
center_lon = sim_cfg.center_lon if sim_cfg else 12.0417

configured_locations = list(sim_cfg.locations) if sim_cfg and sim_cfg.locations else []

if configured_locations:
    entries = [
        {"entry_id": loc.location_id, "lat": loc.lat, "lon": loc.lon}
        for loc in configured_locations
    ]
else:
    entries = [
        {"entry_id": "E1", "lat": center_lat + 0.0014, "lon": center_lon - 0.0009},
        {"entry_id": "E2", "lat": center_lat + 0.0012, "lon": center_lon + 0.0011},
        {"entry_id": "E3", "lat": center_lat - 0.0012, "lon": center_lon + 0.0010},
        {"entry_id": "E4", "lat": center_lat - 0.0013, "lon": center_lon - 0.0011},
    ]

exit_waypoints = [
    {"waypoint_id": "X1", "lat": center_lat + 0.0030, "lon": center_lon - 0.0030},
    {"waypoint_id": "X2", "lat": center_lat + 0.0030, "lon": center_lon + 0.0030},
    {"waypoint_id": "X3", "lat": center_lat - 0.0030, "lon": center_lon + 0.0030},
    {"waypoint_id": "X4", "lat": center_lat - 0.0030, "lon": center_lon - 0.0030},
]

name_pool = [
    "Alex", "Maja", "Jonas", "Emma", "Noah", "Ida", "Lucas", "Freja", "Oscar", "Sofia",
    "Viktor", "Alma", "William", "Clara", "Malthe", "Nora", "Anton", "Liva", "Elias", "Agnes"
]

print(f"Entries configured: {len(entries)}")
print(f"Exit waypoints configured: {len(exit_waypoints)}")
print(f"Using random seed: {SEED}")
print(f"Configured total people: {TOTAL_PEOPLE}")
print(f"Configured tick interval (s): {TICK_SECONDS}")

Entries configured: 4
Exit waypoints configured: 4
Using random seed: 42
Configured total people: 50
Configured tick interval (s): 1.0


In [4]:
# Cell purpose: define movement helpers (distance, step movement, nearest target lookups)
def meters_between(lat1: float, lon1: float, lat2: float, lon2: float) -> float:
    earth_radius_m = 6_371_000.0
    phi1 = math.radians(lat1)
    phi2 = math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlambda = math.radians(lon2 - lon1)
    a = (
        math.sin(dphi / 2) ** 2
        + math.cos(phi1) * math.cos(phi2) * math.sin(dlambda / 2) ** 2
    )
    c = 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))
    return earth_radius_m * c

def move_towards(lat: float, lon: float, target_lat: float, target_lon: float, step_m: float) -> tuple[float, float]:
    distance_m = meters_between(lat, lon, target_lat, target_lon)
    if distance_m <= 1e-9:
        return lat, lon
    ratio = min(1.0, step_m / distance_m)
    new_lat = lat + (target_lat - lat) * ratio
    new_lon = lon + (target_lon - lon) * ratio
    return new_lat, new_lon

def nearest_point(lat: float, lon: float, points: list[dict], id_key: str) -> dict:
    return min(points, key=lambda point: (meters_between(lat, lon, point["lat"], point["lon"]), point[id_key]))

print("Movement helper functions are ready.")

Movement helper functions are ready.


In [5]:
# Cell purpose: initialize 50 people with white status and dynamic nearest-entry routing
rng = random.Random(SEED)
people: list[PersonState] = []

for index in range(TOTAL_PEOPLE):
    start_lat = center_lat + rng.uniform(-0.0022, 0.0022)
    start_lon = center_lon + rng.uniform(-0.0022, 0.0022)
    nearest_entry = nearest_point(start_lat, start_lon, entries, "entry_id")
    nearest_exit = nearest_point(start_lat, start_lon, exit_waypoints, "waypoint_id")

    person = PersonState(
        person_id=f"p{index + 1:03d}",
        name=name_pool[index % len(name_pool)],
        lat=start_lat,
        lon=start_lon,
        state="approaching_entry",
        color="white",
        speed_mps=rng.uniform(0.8, 1.8),
        target_entry_id=nearest_entry["entry_id"],
        target_exit_id=nearest_exit["waypoint_id"],
    )
    people.append(person)

print(f"Initialized {len(people)} people.")
print(f"Initial white count: {sum(1 for person in people if person.color == 'white')}")

Initialized 50 people.
Initial white count: 50


In [6]:
# Cell purpose: connect to MQTT broker and define Phase 3 publish topics
client = mqtt.connect_mqtt(mqtt_cfg, client_id_suffix="agent-people-phase3")

person_state_topic = f"{mqtt_cfg.base_topic}/person/state"
entry_event_topic = f"{mqtt_cfg.base_topic}/entry/event"

print(f"Connected to MQTT broker at {mqtt_cfg.host}:{mqtt_cfg.port}")
print(f"Publishing person state to: {person_state_topic}")
print(f"Publishing entry events to: {entry_event_topic}")

Connected to MQTT broker at broker.mqttdashboard.com:1883
Publishing person state to: simulated-city/stadium/person/state
Publishing entry events to: simulated-city/stadium/entry/event


In [7]:
# Cell purpose: run simulation loop and publish Phase 3 MQTT messages (state + entry events)
simulation_log: list[dict] = []
inside_count = 0
state_publish_success = 0
state_publish_fail = 0
entry_event_publish_success = 0
entry_event_publish_fail = 0

for tick in range(TICKS):
    tick_ts = datetime.now(timezone.utc).isoformat()

    for person in people:
        if person.state == "inside" or person.state == "exited":
            continue

        if person.state == "approaching_entry":
            entry = next(entry_item for entry_item in entries if entry_item["entry_id"] == person.target_entry_id)
            person.lat, person.lon = move_towards(
                person.lat, person.lon, entry["lat"], entry["lon"], person.speed_mps * TICK_SECONDS
            )
            distance_to_entry = meters_between(person.lat, person.lon, entry["lat"], entry["lon"])

            if distance_to_entry <= ENTRY_PROXIMITY_M:
                is_allowed = rng.random() < TRUE_ALLOW_PROBABILITY
                if is_allowed:
                    person.state = "inside"
                    person.color = "green"
                    inside_count += 1
                    event_type = "entered"
                else:
                    person.state = "exiting"
                    person.color = "red"
                    event_type = "denied"

                entry_event_payload = {
                    "person_id": person.person_id,
                    "timestamp": tick_ts,
                    "event_type": event_type,
                    "entry_id": person.target_entry_id,
                }
                if mqtt.publish_json_checked(client, entry_event_topic, entry_event_payload, qos=0):
                    entry_event_publish_success += 1
                else:
                    entry_event_publish_fail += 1

        elif person.state == "exiting":
            exit_point = next(point for point in exit_waypoints if point["waypoint_id"] == person.target_exit_id)
            person.lat, person.lon = move_towards(
                person.lat, person.lon, exit_point["lat"], exit_point["lon"], person.speed_mps * TICK_SECONDS
            )
            distance_to_exit = meters_between(person.lat, person.lon, exit_point["lat"], exit_point["lon"])
            if distance_to_exit <= EXIT_REACHED_M:
                person.state = "exited"
                exit_event_payload = {
                    "person_id": person.person_id,
                    "timestamp": tick_ts,
                    "event_type": "exited",
                    "entry_id": person.target_entry_id,
                }
                if mqtt.publish_json_checked(client, entry_event_topic, exit_event_payload, qos=0):
                    entry_event_publish_success += 1
                else:
                    entry_event_publish_fail += 1

        person_state_payload = {
            "person_id": person.person_id,
            "name": person.name,
            "timestamp": tick_ts,
            "lat": person.lat,
            "lon": person.lon,
            "state": person.state,
            "color": person.color,
            "speed_mps": person.speed_mps,
            "target_entry_id": person.target_entry_id,
        }
        if mqtt.publish_json_checked(client, person_state_topic, person_state_payload, qos=0):
            state_publish_success += 1
        else:
            state_publish_fail += 1

    white_count = sum(1 for person in people if person.color == "white")
    green_count = sum(1 for person in people if person.color == "green")
    red_count = sum(1 for person in people if person.color == "red")

    simulation_log.append(
        {
            "tick": tick,
            "white": white_count,
            "green": green_count,
            "red": red_count,
            "inside_count": inside_count,
        }
    )

final_white = simulation_log[-1]["white"]
final_green = simulation_log[-1]["green"]
final_red = simulation_log[-1]["red"]
final_exited = sum(1 for person in people if person.state == "exited")

print("Phase 3 simulation + publishing complete.")
print(f"Final counts -> white={final_white}, green={final_green}, red={final_red}, exited={final_exited}, inside_count={inside_count}")
print(f"State publish results -> success={state_publish_success}, fail={state_publish_fail}")
print(f"Entry event publish results -> success={entry_event_publish_success}, fail={entry_event_publish_fail}")
print("Sample records (first 3 people):")
for person in people[:3]:
    print({
        "person_id": person.person_id,
        "name": person.name,
        "state": person.state,
        "color": person.color,
        "target_entry_id": person.target_entry_id,
    })

Phase 3 simulation + publishing complete.
Final counts -> white=0, green=39, red=11, exited=3, inside_count=39
State publish results -> success=3898, fail=0
Entry event publish results -> success=53, fail=0
Sample records (first 3 people):
{'person_id': 'p001', 'name': 'Alex', 'state': 'inside', 'color': 'green', 'target_entry_id': 'E1'}
{'person_id': 'p002', 'name': 'Maja', 'state': 'exited', 'color': 'red', 'target_entry_id': 'E3'}
{'person_id': 'p003', 'name': 'Jonas', 'state': 'inside', 'color': 'green', 'target_entry_id': 'E1'}
